In [ ]:
!pip install pandas
!pip install transformers sentencepiece torch --quiet

In [ ]:
import os
from google.colab import userdata

# API keys are set via Colab secrets as needed

In [ ]:
# ============ CELL 2: Imports (FIXED) ============
import torch
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW

from transformers import T5Tokenizer, T5ForConditionalGeneration
from transformers import get_linear_schedule_with_warmup

import pandas as pd
import numpy as np
import time
import random

# Set random seeds for reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True

print(f"PyTorch version: {torch.__version__}")
print(f"GPU available: {torch.cuda.is_available()}")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
# Cell 1: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

SUMMARIZATION_MODEL_PATH = "/content/drive/MyDrive/review_analysis/models/summarization_model"
DATA_PATH = "/content/drive/MyDrive/review_analysis/data"

Mounted at /content/drive


In [ ]:
train_df = pd.read_csv(f'{DATA_PATH}/combined_summarization_final.csv')

val_df = pd.read_csv(f'{DATA_PATH}/validation_data.csv')

print(f"Training samples: {len(train_df)}")
print(f"Validation samples: {len(val_df)}")

Training samples: 156
Validation samples: 48


In [ ]:
MODEL_NAME = "t5-small"

tokenizer = T5Tokenizer.from_pretrained(MODEL_NAME)
model = T5ForConditionalGeneration.from_pretrained(MODEL_NAME)
model = model.to(device)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/242M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [ ]:
MAX_INPUT_LENGTH = 512
MAX_TARGET_LENGTH = 128
BATCH_SIZE = 4
EPOCHS = 10
LEARNING_RATE = 3e-5
PREFIX = "summarize: "

In [ ]:
class SummarizationDataset(Dataset):
    def __init__(self, reviews, summaries, tokenizer, max_input_len, max_target_len):
        self.reviews = reviews
        self.summaries = summaries
        self.tokenizer = tokenizer
        self.max_input_len = max_input_len
        self.max_target_len = max_target_len

    def __len__(self):
        return len(self.reviews)

    def __getitem__(self, idx):
        review = PREFIX + str(self.reviews[idx])
        summary = str(self.summaries[idx])

        # Tokenize input
        input_encoding = self.tokenizer(
            review,
            max_length=self.max_input_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )

        # Tokenize target
        target_encoding = self.tokenizer(
            summary,
            max_length=self.max_target_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )

        labels = target_encoding["input_ids"]
        labels[labels == self.tokenizer.pad_token_id] = -100  # Ignore padding in loss

        return {
            "input_ids": input_encoding["input_ids"].flatten(),
            "attention_mask": input_encoding["attention_mask"].flatten(),
            "labels": labels.flatten()
        }

In [ ]:
train_dataset = SummarizationDataset(
    train_df["reviews"].tolist(),
    train_df["summary"].tolist(),
    tokenizer,
    MAX_INPUT_LENGTH,
    MAX_TARGET_LENGTH
)

val_dataset = SummarizationDataset(
    val_df["reviews"].tolist(),
    val_df["summary"].tolist(),
    tokenizer,
    MAX_INPUT_LENGTH,
    MAX_TARGET_LENGTH
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"Train batches: {len(train_loader)}")
print(f"Val batches: {len(val_loader)}")

Train batches: 39
Val batches: 12


In [ ]:
optimizer = AdamW(model.parameters(), lr=LEARNING_RATE)

num_train_steps = len(train_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=0,
    num_training_steps=num_train_steps
)

In [ ]:
best_val_loss = float("inf")
history = {"train_loss": [], "val_loss": []}
PATIENCE = 3
patience_counter = 0

print("Starting training...")
print(f"Epochs: {EPOCHS}, Batch size: {BATCH_SIZE}, Patience: {PATIENCE}")
print("=" * 50)

for epoch in range(EPOCHS):
    print(f"\nEpoch {epoch + 1}/{EPOCHS}")
    start_time = time.time()

    # ===== Training =====
    model.train()
    train_losses = []

    for batch in train_loader:
        optimizer.zero_grad()

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )

        loss = outputs.loss
        loss.backward()
        optimizer.step()
        scheduler.step()

        train_losses.append(loss.item())

    # ===== Validation =====
    model.eval()
    val_losses = []

    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels
            )

            val_losses.append(outputs.loss.item())

    # Calculate averages
    train_loss = np.mean(train_losses)
    val_loss = np.mean(val_losses)

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)

    epoch_time = time.time() - start_time

    print(f"  Train Loss: {train_loss:.4f}")
    print(f"  Val Loss:   {val_loss:.4f}")
    print(f"  Time:       {epoch_time:.2f}s")

    # Save best model + early stopping
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_counter = 0
        torch.save(model.state_dict(), "best_model.pt")
        tokenizer.save_pretrained("./best_model_tokenizer")
        print("  ✓ Best model saved!")
    else:
        patience_counter += 1
        print(f"  No improvement ({patience_counter}/{PATIENCE})")
        if patience_counter >= PATIENCE:
            print(f"\n  ⚠ Early stopping triggered after {epoch + 1} epochs")
            break

print("\n" + "=" * 50)
print(f"Training complete! Best val loss: {best_val_loss:.4f}")

In [ ]:
model.load_state_dict(torch.load("best_model.pt", map_location=device))
model.eval()
print("✓ Best model loaded!")

In [ ]:
def summarize(reviews):
    """
    Generate summary from reviews

    Args:
        reviews: string or list of strings

    Returns:
        summary string
    """
    # Combine if list
    if isinstance(reviews, list):
        text = " ".join(reviews)
    else:
        text = reviews

    # Prepare input
    input_text = PREFIX + text

    inputs = tokenizer(
        input_text,
        max_length=MAX_INPUT_LENGTH,
        padding="max_length",
        truncation=True,
        return_tensors="pt"
    ).to(device)

    # Generate
    with torch.no_grad():
        outputs = model.generate(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            max_length=MAX_TARGET_LENGTH,
            num_beams=4,
            length_penalty=2.0,
            early_stopping=True,
            no_repeat_ngram_size=2
        )

    # Decode
    summary = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return summary

In [ ]:
print("Testing summarization...")
print("=" * 50)

# Test 1
test1 = "My experience with Events and Adventures was very disappointing. The staff didn't seem very helpful or interested in resolving our issuesand getting accurate event details was frustrating. Customer support took a long time to respond and didn’t really offer any clear solutions. Overall, the whole process felt disorganized. I expected a much smoother and more reliable experience from a site that handles event listings and ticketing."
print(f"\nInput: {test1}")
print(f"Summary: {summarize(test1)}")

# Test 2
test2 = [
    "Parking was terrible. Had to walk 20 minutes.",
    "Sound system kept failing during the concert.",
    "Food ran out within the first hour."
]
print(f"\nInput: {test2}")
print(f"Summary: {summarize(test2)}")


Testing summarization...

Input: My experience with Events and Adventures was very disappointing. The staff didn't seem very helpful or interested in resolving our issuesand getting accurate event details was frustrating. Customer support took a long time to respond and didn’t really offer any clear solutions. Overall, the whole process felt disorganized. I expected a much smoother and more reliable experience from a site that handles event listings and ticketing.
Summary: Events and Adventures: disappointing experience, lack of customer support, and disorganized process. site handles event listings, ticketing, etc.

Input: ['Parking was terrible. Had to walk 20 minutes.', 'Sound system kept failing during the concert.', 'Food ran out within the first hour.']
Summary: Parking was terrible, had to walk 20 minutes, and sound system kept failing during the concert. Food ran out within the first hour.


In [ ]:
model.save_pretrained(SUMMARIZATION_MODEL_PATH)
tokenizer.save_pretrained(SUMMARIZATION_MODEL_PATH)

print(f"Model saved to {SUMMARIZATION_MODEL_PATH}")
print("\nTo load later:")
print("  from transformers import T5ForConditionalGeneration, T5Tokenizer")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved to /content/drive/MyDrive/review_analysis/models/summarization_model

To load later:
  from transformers import T5ForConditionalGeneration, T5Tokenizer


In [ ]:
HF_USERNAME = "MHKaungPyae"

model.push_to_hub(f"{HF_USERNAME}/summarization-model")
tokenizer.push_to_hub(f"{HF_USERNAME}/summarization-model")
print("✓ Summarization model uploaded!")

README.md: 0.00B [00:00, ?B/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...0xxtdcx/model.safetensors:   0%|          |  553kB /  242MB            

No files have been modified since last commit. Skipping to prevent empty commit.


✓ Summarization model uploaded!
